In [1]:
import bw2data, bw2io
import bw2calc
import numpy as np
import pandas as pd
import os
import sys

In [2]:
bw2data.projects.set_current('iveo_v1')

In [3]:
# already 18 pGWP100 created: 
gwps = [m for m in bw2data.methods if 'pGWP' in str(m)] 
len(gwps)

18

In [4]:
m_name = 'Climate Change' + ' prospective GWP100'  # new root name for the 18 pGWP100 
MY = ['2030', '2040', '2050']
SSP = ['SSP119', 'SSP245', 'SSP585']      # match the SSP name defined in the already created pGWP100 

for ss in SSP: 
    for yy in MY:

        # grab the pGW100 fixedCO2 CFs, directly copy all CFs (all float already) to the new method 
        gwp_fixedCO2 =  [m for m in bw2data.methods if 'dpCFs' in str(m) 
                   and  'pGWP100' in str(m) and yy in str(m) and ss in str(m) and 'with fixed AGWPCO2' in str(m)] 

        if len(gwp_fixedCO2) == 1: 
            # create a new method with new name: 
            name1 = (m_name, ss , yy, 'pGWP100 with fixed AGWPCO2') 
            new_method1 = bw2data.Method(name1)
            new_method1.register()
            new_method1.metadata["unit"] = 'kg CO2 2019 eq'
            # re-use the CFs prepared before: 
            cf_list1 = bw2data.Method(gwp_fixedCO2[0]).load()
            # write all CFs to the newly named method
            new_method1.write(cf_list1)
            print(f"{new_method1} written to BW2")

        else: 
            print(f"can't identify unique pGWP100 for SSP{ss}, MY{yy}")
 

        # grab the pGW100 dpCO2 CFs, and copy them (need to change 1 to 1.000 ) to new method  
        gwp_dpCO2 =  [m for m in bw2data.methods if 'dpCFs' in str(m) 
                   and  'pGWP100' in str(m) and yy in str(m) and ss in str(m) 
                      and 'with fixed AGWPCO2' not in str(m)] 

        if len(gwp_dpCO2) == 1: 
            name2 = (m_name, ss , yy, 'pGWP100 with dpAGWPCO2') 
            new_method2 = bw2data.Method(name2)
            new_method2.register() 
            new_method2.metadata["unit"] = 'kg CO2 MY[t]SSP[x] eq'
            # re-use the CFs prepared before: 
            cf_list2 = bw2data.Method(gwp_dpCO2[0]).load()
            # update the old CFs, because old CFs with integer not working for export to json: 
            new_cf_list2 = [
                (flow, 1.000) if cf == 1 else
                (flow, -1.000) if cf == -1 else
                (flow, 0.000) if cf == 0 else
                (flow, float(cf))
                for flow, cf in cf_list2
            ]
            # directly copy all CFs to the newly named method
            new_method2.write(new_cf_list2)
            print(f"{new_method2} written to BW2")

        else:
            print(f"can't identify unique pGWP100 for SSP{ss}, MY{yy}")


Brightway2 Method: Climate Change prospective GWP100: SSP119: 2030: pGWP100 with fixed AGWPCO2 written to BW2
Brightway2 Method: Climate Change prospective GWP100: SSP119: 2030: pGWP100 with dpAGWPCO2 written to BW2
Brightway2 Method: Climate Change prospective GWP100: SSP119: 2040: pGWP100 with fixed AGWPCO2 written to BW2
Brightway2 Method: Climate Change prospective GWP100: SSP119: 2040: pGWP100 with dpAGWPCO2 written to BW2
Brightway2 Method: Climate Change prospective GWP100: SSP119: 2050: pGWP100 with fixed AGWPCO2 written to BW2
Brightway2 Method: Climate Change prospective GWP100: SSP119: 2050: pGWP100 with dpAGWPCO2 written to BW2
Brightway2 Method: Climate Change prospective GWP100: SSP245: 2030: pGWP100 with fixed AGWPCO2 written to BW2
Brightway2 Method: Climate Change prospective GWP100: SSP245: 2030: pGWP100 with dpAGWPCO2 written to BW2
Brightway2 Method: Climate Change prospective GWP100: SSP245: 2040: pGWP100 with fixed AGWPCO2 written to BW2
Brightway2 Method: Climate

### export all 18 pGWP100 as the bw2pacakge 

In [10]:
new_allgwps_ic = [bw2data.Method(ic) for ic in list(bw2data.methods) if ('Climate Change prospective GWP100' in str(ic) )]
print(type(new_allgwps_ic), len(new_allgwps_ic))

<class 'list'> 18


In [11]:
bw2io.BW2Package.export_objs(new_allgwps_ic, 
                            filename = 'pgwp100_N18_3SSP_3MY_2CO2approach',
                            folder='export_pGWP', backwards_compatible=False)

'/Users/susierwu/Library/Application Support/Brightway3/iveo_v1.714ccc91f8f5978fb91e8d2cc7cf50ab/export_pGWP/pgwp100_N18_3SSP_3MY_2CO2approach.141b447f0e14c9d70ddfb10cb23578d6.bw2package'